# DFI Quant — Feature Dashboard
**BTCUSDT · Binance Futures · 2024-03-01**

Sections :
1. Prix + Momentum multi-horizons
2. Microstructure (trade imbalance + realized vol)
3. Matrice de corrélation
4. Distributions des features
5. IC proxy — corrélation de chaque feature avec le retour futur

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 130, 'font.size': 9})

ROOT     = pathlib.Path.home() / 'Downloads' / 'dfi-quant'
RAW      = ROOT / 'data' / 'raw'
FEAT     = ROOT / 'data' / 'features'
EXCHANGE = 'binance-futures'
SYMBOL   = 'BTCUSDT'
DAY      = '2024-03-01'
y, m, d  = DAY.split('-')

def load_feat(fid, family, version='v1'):
    p = (FEAT / f'feature_family={family}' / f'feature_id={fid}'
              / f'version={version}' / f'asset={SYMBOL}'
              / f'year={y}' / f'month={m}' / f'day={d}')
    files = sorted(p.glob('*.parquet'))
    if not files:
        return pd.Series(name=fid, dtype='float64')
    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    df['ts_utc'] = pd.to_datetime(df['ts'], unit='us', utc=True)
    return df.set_index('ts_utc')['value'].rename(fid)

print('ROOT exists:', ROOT.exists())

In [ ]:
# --- Prix 1-min ---
raw_dir = (RAW / f'exchange={EXCHANGE}' / 'data_type=trades'
               / f'symbol={SYMBOL}' / f'year={y}' / f'month={m}' / f'day={d}')
trades  = pd.concat([pd.read_parquet(p) for p in sorted(raw_dir.glob('*.parquet'))],
                    ignore_index=True)
trades['ts_utc'] = pd.to_datetime(trades['ts'], unit='us', utc=True)
px = (trades.set_index('ts_utc')['price']
            .resample('1min', label='right', closed='right')
            .last().ffill())

# --- Features ---
ti   = load_feat('trade_imbalance_5m',     'microstructure')
rv   = load_feat('realized_volatility_5m', 'microstructure')
m15  = load_feat('price_momentum_15m',     'momentum')
m1h  = load_feat('price_momentum_1h',      'momentum')
m4h  = load_feat('price_momentum_4h',      'momentum')

print(f'Trades  : {len(trades):,} ticks')
print(f'Prix    : {len(px)} barres 1-min')
for s in [ti, rv, m15, m1h, m4h]:
    print(f'{s.name:<30} {len(s):>5} barres  '
          f'mean={s.mean():+.5f}  std={s.std():.5f}')

## 1 · Prix & Momentum multi-horizons

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1, 1, 1]})

# Prix
ax = axes[0]
ax.plot(px.index, px.values, color='#1f1f1f', lw=1.0)
ax.set_ylabel('USDT')
ax.set_title(f'BTCUSDT · Binance Futures · {DAY}', fontweight='bold')
ax.grid(alpha=0.25)

# Momentum 15m
ax = axes[1]
c15 = np.where(m15.values >= 0, '#2ecc71', '#e74c3c')
ax.bar(m15.index, m15.values, width=1/1440, color=c15, alpha=0.8)
ax.axhline(0, color='k', lw=0.5)
ax.set_ylabel('mom 15m', fontsize=8)
ax.grid(alpha=0.2)

# Momentum 1h
ax = axes[2]
ax.plot(m1h.index, m1h.values, color='#3498db', lw=0.9)
ax.fill_between(m1h.index, 0, m1h.values,
                where=m1h.values >= 0, color='#3498db', alpha=0.25)
ax.fill_between(m1h.index, 0, m1h.values,
                where=m1h.values < 0,  color='#e74c3c', alpha=0.25)
ax.axhline(0, color='k', lw=0.5)
ax.set_ylabel('mom 1h', fontsize=8)
ax.grid(alpha=0.2)

# Momentum 4h
ax = axes[3]
ax.plot(m4h.index, m4h.values, color='#9b59b6', lw=0.9)
ax.fill_between(m4h.index, 0, m4h.values,
                where=m4h.values >= 0, color='#9b59b6', alpha=0.25)
ax.fill_between(m4h.index, 0, m4h.values,
                where=m4h.values < 0,  color='#e74c3c', alpha=0.25)
ax.axhline(0, color='k', lw=0.5)
ax.set_ylabel('mom 4h', fontsize=8)
ax.set_xlabel('UTC')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 2 · Microstructure — Trade Imbalance & Realized Vol

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True,
                         gridspec_kw={'height_ratios': [2.5, 1.5, 1.5]})

# Prix avec fond coloré par trade imbalance
ax = axes[0]
ax.plot(px.index, px.values, color='#1f1f1f', lw=1.0, zorder=3)
# zones de forte pression acheteuse / vendeuse
ti_aligned = ti.reindex(px.index, method='ffill')
ax.fill_between(px.index, px.min(), px.max(),
                where=ti_aligned > 0.15, color='#2ecc71', alpha=0.12, label='buy pressure > 0.15')
ax.fill_between(px.index, px.min(), px.max(),
                where=ti_aligned < -0.15, color='#e74c3c', alpha=0.12, label='sell pressure < -0.15')
ax.set_ylabel('USDT')
ax.set_title('Microstructure overlay', fontweight='bold')
ax.legend(fontsize=8, loc='upper left')
ax.grid(alpha=0.25)

# Trade imbalance
ax = axes[1]
colors = np.where(ti.values >= 0, '#2ecc71', '#e74c3c')
ax.bar(ti.index, ti.values, width=1/1440, color=colors, alpha=0.8)
ax.axhline(0,     color='k',        lw=0.5)
ax.axhline( 0.15, color='#2ecc71',  lw=0.7, ls='--', alpha=0.6)
ax.axhline(-0.15, color='#e74c3c',  lw=0.7, ls='--', alpha=0.6)
ax.set_ylabel('trade_imbalance_5m')
ax.set_ylim(-0.6, 0.6)
ax.grid(alpha=0.2)

# Realized vol — spikes mis en évidence
ax = axes[2]
vol_thresh = rv.quantile(0.90)
ax.plot(rv.index, rv.values, color='#8e44ad', lw=0.9)
ax.fill_between(rv.index, 0, rv.values, color='#8e44ad', alpha=0.18)
ax.fill_between(rv.index, 0, rv.values,
                where=rv.values > vol_thresh, color='#f39c12', alpha=0.6,
                label=f'vol spike (>p90={vol_thresh:.4f})')
ax.axhline(vol_thresh, color='#f39c12', lw=0.8, ls='--')
ax.set_ylabel('realized_vol_5m')
ax.set_xlabel('UTC')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 3 · Matrice de corrélation entre features

In [ ]:
all_feat = pd.concat([ti, rv, m15, m1h, m4h], axis=1).dropna()
labels   = ['TI 5m', 'RV 5m', 'Mom 15m', 'Mom 1h', 'Mom 4h']
all_feat.columns = labels

corr_p = all_feat.corr(method='pearson')
corr_s = all_feat.corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, corr, title in zip(axes, [corr_p, corr_s], ['Pearson', 'Spearman']):
    im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap='RdYlGn', aspect='auto')
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            v = corr.values[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    fontsize=9, color='black' if abs(v) < 0.6 else 'white')
    ax.set_title(f'{title} — {len(all_feat)} barres communes', fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

## 4 · Distributions des features

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
colors = ['#27ae60', '#8e44ad', '#3498db', '#2980b9', '#6c3483']

for ax, (name, col), c in zip(axes, all_feat.items(), colors):
    vals = col.dropna().values
    ax.hist(vals, bins=60, color=c, alpha=0.65, density=True, edgecolor='none')
    ax.axvline(0,              color='k',      lw=0.7, ls='--')
    ax.axvline(np.median(vals), color='orange', lw=1.2, ls='--',
               label=f'med={np.median(vals):.4f}')
    ax.set_title(name, fontweight='bold')
    ax.legend(fontsize=7)
    ax.set_xlabel('valeur')
    ax.grid(alpha=0.2)

plt.suptitle('Distributions empiriques — 2024-03-01', y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

## 5 · IC proxy — Corrélation avec le retour futur

Pour chaque feature $f_t$, on calcule la corrélation de Spearman avec $r_{t+h}$ (log-retour du prix dans $h$ minutes).  
Une IC positive = la feature **prédit** la direction du retour futur.  
⚠️ 1 jour de données = résultat purement illustratif, pas significatif statistiquement.

In [ ]:
# Retours futurs à différents horizons
horizons = {'15m': 15, '1h': 60, '4h': 240}
fwd = {}
for name, h in horizons.items():
    fwd[f'r+{name}'] = np.log(px / px.shift(-h)).shift(-1)  # shift(-1) : on écrit à la barre t

fwd_df = pd.DataFrame(fwd)

results = []
for feat_name, feat_col in all_feat.items():
    for fwd_name, fwd_col in fwd_df.items():
        joined = pd.concat([feat_col, fwd_col], axis=1).dropna()
        if len(joined) < 30:
            continue
        ic = joined.corr(method='spearman').iloc[0, 1]
        results.append({'feature': feat_name, 'horizon': fwd_name, 'IC': ic})

ic_df = pd.DataFrame(results).pivot(index='feature', columns='horizon', values='IC')
print(ic_df.round(3))

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(ic_df))
w = 0.25
cols_h = list(ic_df.columns)
palette = ['#3498db', '#e67e22', '#8e44ad']

for i, (col, c) in enumerate(zip(cols_h, palette)):
    ax.bar(x + (i - 1) * w, ic_df[col].values, width=w, label=col,
           color=c, alpha=0.8)

ax.axhline(0, color='k', lw=0.7)
ax.set_xticks(x)
ax.set_xticklabels(ic_df.index, rotation=20, ha='right')
ax.set_ylabel('IC (Spearman)')
ax.set_title('IC proxy : corrélation feature_t vs retour futur  (1 jour, illustratif)',
             fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()